In [ ]:
import os, sys
from pathlib import Path
# Allow overriding base dir via environment variable BASE_DIR
env_base = os.environ.get('BASE_DIR')
if env_base:
    base_dir = Path(env_base).resolve()
else:
    # Search upward from current working directory for a folder that contains 'module'
    p = Path.cwd().resolve()
    found = None
    for _ in range(20):
        if (p / 'module').is_dir():
            found = p
            break
        if p.parent == p:
            break
        p = p.parent
    base_dir = found or Path.cwd().resolve()
# Ensure base_dir is on sys.path so 'module' can be imported
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))
print(f"Using base_dir: {base_dir}")
import pandas as pd
df = pd.read_csv(str(base_dir / 'data' / 'data.csv'))
df.head()

: 

In [ ]:
%pip install langchain langchain-community langchain-openai

In [ ]:
import pandas as pd
import re

# 1. Đọc dữ liệu từ file data.csv
df = pd.read_csv("../data/data.csv")

# 2. Ghi nhận số lượng dòng trước khi lọc
count_before = len(df)

# 3. Lọc trùng dựa trên cột 'content'
# subset=['content'] : chỉ xét trùng ở cột nội dung
# keep='first' : giữ lại bản ghi đầu tiên xuất hiện
df_cleaned = df.drop_duplicates(subset=['content'], keep='first').copy()

# 4. Hàm kiểm tra và mã hóa cột 'contact'
def contact_has_phone(val):
    if pd.isna(val) or val == '[]':
        return 0
    # Chuyển về chuỗi và lọc lấy các chữ số
    s = str(val)
    digits = re.sub(r'[^0-9]', '', s)
    # Nếu chuỗi số có độ dài từ 7 chữ số trở lên thì coi là có số điện thoại
    return 1 if len(digits) >= 7 else 0

# Áp dụng mã hóa cho cột 'contact'
if 'contact' in df_cleaned.columns:
    df_cleaned['contact'] = df_cleaned['contact'].apply(contact_has_phone)
else:
    df_cleaned['contact'] = 0

# 5. Ghi nhận số lượng dòng sau khi lọc
count_after = len(df_cleaned)

# 6. Lưu kết quả ra file mới (Sử dụng utf-8-sig để đọc được tiếng Việt trong Excel)
df_cleaned.to_csv('data_cleaned.csv', index=False, encoding='utf-8-sig')

# 7. In kết quả kiểm tra
print(f"Số dòng ban đầu: {count_before}")
print(f"Số dòng sau khi lọc trùng: {count_after}")
print(f"Đã xóa: {count_before - count_after} dòng trùng lặp.")
print("\n--- 5 dòng đầu tiên của dữ liệu đã sạch (contact đã được mã hóa) ---")
df_cleaned.head()

In [ ]:
import pandas as pd
import json

from pydantic import BaseModel
from module.agent import Agent

agent = Agent()

# --- PROMPT VÀ XỬ LÝ (SYNCHRONOUS) ---
prompt_template = """
Bạn là một trợ lý phân tích dữ liệu cứu hộ. Hãy bóc tách địa chỉ sau thành định dạng JSON.
Địa chỉ: {location}

Yêu cầu trả về JSON với các trường:
- "tinh_tp": Tỉnh hoặc Thành phố trực thuộc trung ương.
- "quan_huyen": Quận, Huyện hoặc Thị xã.
- "phuong_xa": Phường, Xã hoặc Thị trấn.
- "nha": Số nhà, ngõ, tên đường hoặc các mô tả chi tiết vị trí nhỏ hơn xã.

Chỉ trả về duy nhất khối JSON, không giải thích gì thêm.
Kết quả JSON:
"""
class Response(BaseModel):
    tinh_tp: str
    quan_huyen: str
    phuong_xa: str
    nha: str

def parse_address_local(location: str) -> dict:
    if pd.isna(location) or str(location).strip() == "":
        return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}
    try:
        user_prompt = prompt_template.format(location=location)
        # synchronous call to the agent
        message = agent.generate_sync(user_prompt=user_prompt, response_schema=Response)
        print(message)
        if hasattr(message, 'dict'):
            return message.dict()
        return dict(message)
    except Exception as e:
        print(f"Lỗi dòng: {location} -> {e}")
        return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}

# --- CHẠY TRÊN FILE DATA_CLEANED.CSV ---
df = pd.read_csv('data_cleaned.csv')

print(f"Bắt đầu xử lý {len(df)} dòng địa chỉ...")

# Áp dụng hàm bóc tách một cách tuần tự (synchronous)
results = [parse_address_local(loc) for loc in df['location'].tolist()]

# Chuyển list các dictionary thành các cột riêng biệt
df_address_cols = pd.json_normalize(results)

# Gộp vào dataframe gốc
df_final = pd.concat([df, df_address_cols], axis=1)

# Lưu kết quả
df_final.to_csv('data_encoded_local.csv', index=False, encoding='utf-8-sig')

print("Hoàn thành! Kiểm tra file: data_encoded_local.csv")

In [1]:
import pandas as pd 
df_encoded_local = pd.read_csv("data_encoded_local.csv")
df_encoded_local.head()

,stt,content,reactions,replies,posted_time,location,contact,disaster_type,affected_people,infrastructure_damage,...,deathly,has_emergency_terms,has_emotional_terms,has_emphasis,label,confident,tinh_tp,quan_huyen,phuong_xa,nha
0,1,"CẦN HỖ TRỢ LƯƠNG THỰC THỰC PHẨM, SẠC, ĐÈN PIN ...",82,29,1w,"nhà số 9 kiệt 34 An Ninh, phường Kim Long, thà...",1,lũ lụt,4.0,False,...,False,False,False,True,CAO,CAO,Thừa Thiên Huế,Huế,Kim Long,nhà số 9 kiệt 34 An Ninh
1,2,"CẦN CỨU KhẨN CẤP Khu vực dân Vĩnh Điện, Điện B...",0,0,1w,"Vĩnh Điện, Điện Bàn, Quảng Nam",0,lũ lụt,NaN,True,...,True,True,True,False,CAO,CAO,Quảng Nam,Điện Bàn,Vĩnh Điện,NaN
2,4,"-Người thân em đang ở địa chỉ định vị này,sáng...",0,0,1w,"thôn Tây An, xã Gò Nổi, Đà Nẵng (Quảng Nam cũ)",1,lũ lụt,NaN,True,...,True,False,False,True,CAO,CAO,Quảng Nam,Đà Nẵng,Gò Nổi,thôn Tây An
3,5,"Anh chồng mình là Nguyễn Xuân Năm, bị ngộ độc ...",0,0,1w,quán dê Phố Núi,0,lũ lụt,1.0,False,...,True,False,False,True,CAO,THẤP,Không rõ,Phố Núi,Không rõ,quán dê
4,6,"Đào Thị Quy. Nhà có 1 bé 4 tuổi, 1 bé 6 tuổi, ...",1,0,17h,"đội 9, thôn Hòa Nghĩa, Xã Mỹ Hiệp, huyện Phù M...",1,lũ lụt,6.0,True,...,True,True,False,False,CAO,CAO,Bình Định,Phù Mỹ,Mỹ Hiệp,"đội 9, thôn Hòa Nghĩa"


In [4]:
import pandas as pd
import re

# 1. Đọc dữ liệu
df = pd.read_csv('data_encoded_local.csv') # Hoặc tên file của bạn

# 2. Hàm định nghĩa trọng số mới
def analyze_disaster_weighted_v2(text):
    if pd.isna(text) or str(text).strip() == '':
        keys = ['lũ lụt', 'sạt lở', 'sập nhà', 'thiên tai', 'hỏa hoạn', 'sự cố hạ tầng']
        return {k: 0 for k in keys}, 0, 'không rõ'
    
    text = str(text).lower()
    text = text.replace('sạc lỡ', 'sạt lở').replace('sạc lở', 'sạt lở')
    
    # --- CẬP NHẬT TRỌNG SỐ TẠI ĐÂY ---
    weights_map = {
        'lũ lụt': 5,
        'sạt lở': 5,
        'sập nhà': 5,
        'hỏa hoạn': 5,        # Đã nâng lên mức cao nhất
        'thiên tai': 3,
        'sự cố hạ tầng': 1
    }
    
    found_types = set()
    
    # Logic phát hiện từ khóa
    if re.search(r'lũ|ngập|nước dâng', text): found_types.add('lũ lụt')
    if re.search(r'sạt|trượt|đất', text): found_types.add('sạt lở')
    if re.search(r'sập|đổ|tốc mái', text): found_types.add('sập nhà')
    if re.search(r'hỏa|cháy|nổ', text): found_types.add('hỏa hoạn')
    if re.search(r'điện|mạng|internet|wifi|nước', text): found_types.add('sự cố hạ tầng')
    if re.search(r'thiên tai|bão', text): found_types.add('thiên tai')
    
    encoded_row = {k: 0 for k in weights_map.keys()}
    total_score = 0
    max_weight = 0
    primary_type = 'không rõ'
    
    for dtype in found_types:
        w = weights_map[dtype]
        encoded_row[dtype] = w
        total_score += w
        
        # Xác định loại thiên tai chính (ưu tiên cái có trọng số cao nhất)
        if w > max_weight:
            max_weight = w
            primary_type = dtype
            
    return encoded_row, total_score, primary_type

# 3. Áp dụng vào dữ liệu
results = df['disaster_type'].apply(analyze_disaster_weighted_v2)

# 4. Tách kết quả thành các cột
weighted_df = pd.DataFrame([res[0] for res in results])
weighted_df.columns = [f"w_{c}" for c in weighted_df.columns] # Thêm tiền tố w_

df['disaster_severity_score'] = [res[1] for res in results]
df['primary_disaster_type'] = [res[2] for res in results]

# 5. Gộp vào bảng chính và lưu
df_final = pd.concat([df, weighted_df], axis=1)
df_final.to_csv('data_encoded_weighted_v2.csv', index=False)
